## Generating statistics to make the script more realistic

In [22]:
import os
import json
import numpy as np
from collections import defaultdict
from typing import Dict, List, Any

In [23]:
def as_list_on_duplicate_keys(ordered_pairs):
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list): d[k].append(v)
            else: d[k] = [d[k], v]
        else: d[k] = v
    return d

In [24]:
def calculate_statistics_from_captures(directory_path: str) -> dict:
    """
    Analyzes QUIC capture JSON files to build comprehensive statistical profiles.
    
    Extracts statistics for:
    - Packet sizes (handshake, data, ACKs, control frames, HTTP/3 streams)
    - Delta times (handshake, data transfer, ACKs)
    - ACK frequency patterns
    - Path validation (PATH_CHALLENGE/RESPONSE)
    - Connection migration timing
    
    Args:
        directory_path: Path to directory containing Wireshark JSON exports
        
    Returns:
        Dictionary with statistical profiles (mean, std, sample count)
    """
    raw_stats = defaultdict(lambda: defaultdict(list))
    
    print(f"Analyzing JSON files in: {directory_path}\n")
    
    for filename in os.listdir(directory_path):
        if not filename.endswith('.json'):
            continue
            
        json_file_path = os.path.join(directory_path, filename)
        print(f"  -> Processing: {filename}")
        
        # Load JSON with encoding fallback
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
        except (json.JSONDecodeError, UnicodeDecodeError):
            try:
                with open(json_file_path, 'r', encoding='utf-16') as f:
                    packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
            except Exception as e:
                print(f"     ... Skipping, could not decode JSON: {e}")
                continue
        
        if not packets:
            continue
        
        # Extract initial connection parameters
        try:
            first_packet_layers = packets[0]['_source']['layers']
            if not ('ip' in first_packet_layers and 'frame' in first_packet_layers):
                continue
                
            initial_ip_client = first_packet_layers['ip']['ip.src']
            initial_ip_server = first_packet_layers['ip']['ip.dst']
            initial_port_client = int(first_packet_layers['udp']['udp.srcport'])
            initial_port_server = int(first_packet_layers['udp']['udp.dstport'])
            last_packet_time = float(first_packet_layers['frame']['frame.time_epoch'])
        except (KeyError, IndexError):
            print(f"     ... Skipping, missing required fields")
            continue
        
        # State tracking
        in_handshake = True
        migrated = False
        handshake_done_seen = False
        client_data_since_server_ack = 0
        server_data_since_client_ack = 0
        
        # Process each packet
        for pkt_index, pkt_data in enumerate(packets):
            layers = pkt_data.get('_source', {}).get('layers', {})
            if not ('ip' in layers and 'frame' in layers):
                continue
            
            # Extract packet metadata
            src_ip = layers['ip']['ip.src']
            dst_ip = layers['ip']['ip.dst']
            src_port = int(layers['udp']['udp.srcport'])
            dst_port = int(layers['udp']['udp.dstport'])
            
            # Detect migration
            if not migrated and \
               (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
               (src_ip != initial_ip_client or src_port != initial_port_client):
                migrated = True
                raw_stats['behavior_counts']['packets_before_migration'].append(pkt_index)
            
            # Calculate timing
            current_time = float(layers['frame']['frame.time_epoch'])
            delta_time = current_time - last_packet_time
            last_packet_time = current_time
            
            # Direction
            is_client_pkt = (src_ip == initial_ip_client and src_port == initial_port_client)
            direction = 'client' if is_client_pkt else 'server'
            
            # Packet length
            pkt_len = int(layers['frame']['frame.len'])
            
            # Extract QUIC layer
            quic_packet_list = layers.get('quic', [])
            if not isinstance(quic_packet_list, list):
                quic_packet_list = [quic_packet_list]
            
            # Process each QUIC packet (can have multiple per UDP packet)
            for quic_packet in quic_packet_list:
                # Get packet type
                packet_type = quic_packet.get('quic.long.packet_type')
                is_long_header = quic_packet.get('quic.header_form') == '1'
                
                # Extract frames
                quic_frames = quic_packet.get('quic.frame', [])
                if not isinstance(quic_frames, list):
                    quic_frames = [quic_frames]
                
                # Analyze frame types
                frame_types = {f.get('quic.frame_type', '0') for f in quic_frames}
                
                # Frame type checks
                has_crypto = '0x0000000000000006' in frame_types
                has_ack = '0x0000000000000002' in frame_types or '0x0000000000000003' in frame_types
                has_stream = any('0x0000000000000008' <= ft <= '0x000000000000000f' for ft in frame_types)
                has_padding = '0x0000000000000000' in frame_types
                has_ping = '0x0000000000000001' in frame_types
                has_connection_close = '0x000000000000001c' in frame_types or '0x000000000000001d' in frame_types
                has_path_challenge = '0x000000000000001a' in frame_types
                has_path_response = '0x000000000000001b' in frame_types
                has_new_connection_id = '0x0000000000000018' in frame_types
                has_handshake_done = '0x000000000000001e' in frame_types
                
                # Detect handshake completion
                if has_handshake_done:
                    handshake_done_seen = True
                    in_handshake = False
                
                # ============================================================
                # HANDSHAKE PHASE STATISTICS
                # ============================================================
                if in_handshake or (is_long_header and has_crypto):
                    # Delta times
                    delta_key = 'handshake_c2s' if is_client_pkt else 'handshake_s2c'
                    raw_stats['delta_times'][delta_key].append(delta_time)
                    
                    # Packet sizes
                    if packet_type == '0' and is_client_pkt:
                        # Initial packet from client
                        raw_stats['packet_sizes']['handshake_initial_client'].append(pkt_len)
                    elif packet_type == '0' and not is_client_pkt:
                        # Initial packet from server
                        raw_stats['packet_sizes']['handshake_initial_server'].append(pkt_len)
                    elif packet_type == '2':
                        # Handshake packet
                        raw_stats['packet_sizes'][f'handshake_handshake_{direction}'].append(pkt_len)
                    elif packet_type == '3':
                        # Retry packet
                        raw_stats['packet_sizes']['handshake_retry'].append(pkt_len)
                    else:
                        # Other handshake packets
                        raw_stats['packet_sizes'][f'handshake_other_{direction}'].append(pkt_len)
                
                # ============================================================
                # 1-RTT PHASE STATISTICS
                # ============================================================
                elif not in_handshake or handshake_done_seen:
                    # PATH_CHALLENGE/RESPONSE (connection migration)
                    if has_path_challenge:
                        raw_stats['packet_sizes']['path_challenge'].append(pkt_len)
                        raw_stats['delta_times']['path_challenge'].append(delta_time)
                    
                    if has_path_response:
                        raw_stats['packet_sizes']['path_response'].append(pkt_len)
                        raw_stats['delta_times']['path_response'].append(delta_time)
                    
                    # ACK-only packets
                    if has_ack and not has_stream and not has_connection_close and not has_path_challenge and not has_path_response:
                        raw_stats['packet_sizes'][f'ack_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['ack_response'].append(delta_time)
                        
                        # Track ACK frequency
                        if is_client_pkt:
                            if server_data_since_client_ack > 0:
                                raw_stats['ack_frequency']['client_sends_ack_after'].append(server_data_since_client_ack)
                            server_data_since_client_ack = 0
                        else:
                            if client_data_since_server_ack > 0:
                                raw_stats['ack_frequency']['server_sends_ack_after'].append(client_data_since_server_ack)
                            client_data_since_server_ack = 0
                    
                    # PING-only packets
                    elif has_ping and not has_stream and not has_connection_close:
                        raw_stats['packet_sizes'][f'ping_{direction}'].append(pkt_len)
                        raw_stats['delta_times'][f'ping_{direction}'].append(delta_time)
                    
                    # CONNECTION_CLOSE packets
                    elif has_connection_close:
                        raw_stats['packet_sizes'][f'close_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['close'].append(delta_time)
                    
                    # STREAM packets (application data)
                    elif has_stream:
                        delta_key = 'client_request' if is_client_pkt else 'server_response'
                        raw_stats['delta_times'][delta_key].append(delta_time)
                        
                        # Track for ACK frequency
                        if is_client_pkt:
                            client_data_since_server_ack += 1
                        else:
                            server_data_since_client_ack += 1
                        
                        # Categorize by stream type (based on stream ID)
                        stream_type_found = False
                        for frame in quic_frames:
                            frame_type = frame.get('quic.frame_type', '0')
                            if '0x0000000000000008' <= frame_type <= '0x000000000000000f':
                                stream_id = frame.get('quic.stream.stream_id')
                                
                                if stream_id is not None:
                                    try:
                                        stream_id = int(stream_id)
                                        
                                        # Determine stream type from stream ID
                                        # Stream ID modulo 4 determines initiator and directionality
                                        if stream_id % 4 == 0:
                                            stream_type_key = 'pkt_size_bidi_client'
                                        elif stream_id % 4 == 1:
                                            stream_type_key = 'pkt_size_bidi_server'
                                        elif stream_id % 4 == 2:
                                            stream_type_key = 'pkt_size_uni_client'
                                        elif stream_id % 4 == 3:
                                            stream_type_key = 'pkt_size_uni_server'
                                        
                                        raw_stats['packet_sizes'][stream_type_key].append(pkt_len)
                                        stream_type_found = True
                                        break  # Use first stream in packet
                                    except (ValueError, TypeError):
                                        continue
                        
                        # Fallback if no stream ID found
                        if not stream_type_found:
                            raw_stats['packet_sizes'][f'stream_data_{direction}'].append(pkt_len)
                    
                    # Mixed packets (have ACK + data)
                    elif has_ack and has_stream:
                        # Already counted in stream category above
                        pass
    
    # ============================================================
    # COMPUTE FINAL STATISTICS
    # ============================================================
    final_stats = defaultdict(dict)
    
    for category, keys in raw_stats.items():
        for key, data_list in keys.items():
            # Filter out invalid values
            if 'ack_frequency' in category:
                data_list = [x for x in data_list if x > 0]
            
            if 'delta_times' in category:
                data_list = [x for x in data_list if x >= 0]
            
            if 'packet_sizes' in category:
                data_list = [x for x in data_list if x > 0]
            
            # Calculate statistics
            if data_list and len(data_list) > 0:
                final_stats[category][key] = {
                    'mean': float(np.mean(data_list)),
                    'std': float(np.std(data_list)),
                    'min': float(np.min(data_list)),
                    'max': float(np.max(data_list)),
                    'median': float(np.median(data_list)),
                    'samples': len(data_list)
                }
            else:
                final_stats[category][key] = {
                    'mean': 0.0,
                    'std': 0.0,
                    'min': 0.0,
                    'max': 0.0,
                    'median': 0.0,
                    'samples': 0
                }
    
    print(f"\n=== Statistics Summary ===")
    print(f"Packet size categories: {len(final_stats.get('packet_sizes', {}))}")
    print(f"Delta time categories: {len(final_stats.get('delta_times', {}))}")
    print(f"ACK frequency metrics: {len(final_stats.get('ack_frequency', {}))}")
    print(f"Behavior counts: {len(final_stats.get('behavior_counts', {}))}")
    
    return dict(final_stats)


def print_statistics_summary(stats: Dict[str, Any]) -> None:
    """Pretty print statistics summary."""
    print("\n" + "="*80)
    print("STATISTICS SUMMARY")
    print("="*80)
    
    for category, metrics in stats.items():
        print(f"\n{category.upper()}:")
        print("-" * 80)
        
        for key, values in sorted(metrics.items()):
            if values['samples'] > 0:
                print(f"  {key:40s} | μ={values['mean']:8.4f}  σ={values['std']:8.4f}  "
                      f"n={values['samples']:4d}  [{values['min']:.2f}, {values['max']:.2f}]")
            else:
                print(f"  {key:40s} | No samples")


In [25]:
json_captures_directory = r"C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test"

if not os.path.isdir(json_captures_directory):
    print(f"Error: Directory not found at '{json_captures_directory}'")
    print("Please update the 'json_captures_directory' variable with the correct path.")
else:
    full_stats_profile = calculate_statistics_from_captures(json_captures_directory)
    
    print("\n\n--- Calculated Statistical Profile ---")
    print(json.dumps(full_stats_profile, indent=4))

Analyzing JSON files in: C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test

  -> Processing: 1_quiche_capture.json
  -> Processing: 2_quiche_capture.json
  -> Processing: 3172_aioquic_before_fast.json
  -> Processing: 3173_aioquic_before_fast.json
  -> Processing: 3174_aioquic_before_fast.json
  -> Processing: 3175_aioquic_before_fast.json
  -> Processing: 3941_aioquic_before_slow_600.json
  -> Processing: 3942_aioquic_before_slow_600.json
  -> Processing: 3943_aioquic_before_slow_600.json
  -> Processing: 3944_aioquic_before_slow_600.json
  -> Processing: 3_quiche_capture.json
  -> Processing: 4234_aioquic_during_slow_200.json
  -> Processing: 4235_aioquic_during_slow_200.json
  -> Processing: 4236_aioquic_during_slow_200.json
  -> Processing: 4237_aioquic_during_slow_200.json
  -> Processing: 4918_aioquic_during_slow_600.json
  -> Processing: 4919_aioquic_during_slow_600.json
  -> Processing: 4920_aioquic_during_slow_600.json
  -> Processing: 4921_ai

In [26]:
print_statistics_summary(full_stats_profile)


STATISTICS SUMMARY

DELTA_TIMES:
--------------------------------------------------------------------------------
  ack_response                             | μ=  0.0080  σ=  0.0082  n= 119  [0.00, 0.03]
  client_request                           | μ=  0.0049  σ=  0.0079  n=  16  [0.00, 0.02]
  close                                    | μ=  0.2803  σ=  0.4504  n=  29  [0.00, 1.01]
  handshake_c2s                            | μ=  0.0014  σ=  0.0010  n= 215  [0.00, 0.00]
  handshake_s2c                            | μ=  0.0027  σ=  0.0023  n= 102  [0.00, 0.01]
  path_challenge                           | μ=  0.0006  σ=  0.0003  n=  34  [0.00, 0.00]
  path_response                            | μ=  0.0037  σ=  0.0070  n=  34  [0.00, 0.03]
  ping_server                              | μ=  0.0097  σ=  0.0135  n=  15  [0.00, 0.03]
  server_response                          | μ=  0.0269  σ=  0.0519  n= 161  [0.00, 0.15]

PACKET_SIZES:
------------------------------------------------------------

In [27]:
full_stats_profile

{'delta_times': {'handshake_c2s': {'mean': 0.0013973712921142578,
   'std': 0.000967970295872902,
   'min': 0.0,
   'max': 0.003258943557739258,
   'median': 0.001772165298461914,
   'samples': 215},
  'handshake_s2c': {'mean': 0.0026799814373839135,
   'std': 0.002345290535902211,
   'min': 0.00011301040649414062,
   'max': 0.009875059127807617,
   'median': 0.0031070709228515625,
   'samples': 102},
  'server_response': {'mean': 0.026936246741632495,
   'std': 0.051887949764005736,
   'min': 0.0,
   'max': 0.14530014991760254,
   'median': 0.0008790493011474609,
   'samples': 161},
  'ack_response': {'mean': 0.00799737056764234,
   'std': 0.008152223605089283,
   'min': 0.0,
   'max': 0.029800891876220703,
   'median': 0.007986068725585938,
   'samples': 119},
  'path_challenge': {'mean': 0.0005812644958496094,
   'std': 0.0003131434010756422,
   'min': 0.0001990795135498047,
   'max': 0.0015611648559570312,
   'median': 0.0005314350128173828,
   'samples': 34},
  'path_response': {'

#### Generating low level features with the statistics

In [28]:
pd.set_option('display.max_columns', None)  # Show all columns when printing

In [29]:
import numpy as np
import csv
import pandas as pd
from typing import Dict, List, Tuple, Optional

SIMULATED_MTU = 1350
PATH_VALIDATION_MTU_MIN = 1200  # RFC 9000 minimum for path validation
PATH_VALIDATION_MTU_MAX = 1450  # Allow some variance


def generate_statistically_realistic_features(blueprint: dict, stats: dict) -> list:
    """
    Generates a flexible, statistics-driven sequence of QUIC packets.
    
    This version:
    - Uses statistics for ALL packet sizes (not hardcoded)
    - Adds realistic randomness to delta times based on network conditions
    - Adapts to the high-level blueprint while maintaining protocol correctness
    - Generates varied captures from the same blueprint
    """
    output_packet_features = []
    current_time_msec = 0.0
    current_packet_count = 0
    
    # Track application bytes
    client_app_bytes_sent = 0
    server_app_bytes_sent = 0
    has_migrated = False
    
    # Network jitter simulation (adds realism to delta times)
    base_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 2.0  # Half RTT
    network_jitter = np.random.uniform(0.5, 1.5)  # Random network conditions
    
    def get_stat(category: str, key: str, fallback_mean: float = 100, 
                 fallback_std: float = 20, strict_positive: bool = False) -> float:
        """
        Safely draw from statistics with fallbacks and bounds.
        """
        if category in stats and key in stats[category] and stats[category][key]['samples'] > 0:
            mean = stats[category][key]['mean']
            std = stats[category][key]['std']
            value = np.random.normal(mean, std)
        else:
            value = np.random.normal(fallback_mean, fallback_std)
        
        # Apply bounds
        if 'delta' in key or 'time' in key:
            return max(0.000001, value) * network_jitter
        elif 'size' in key or 'length' in key:
            min_val = 50 if strict_positive else 40
            return int(max(min_val, min(SIMULATED_MTU, value)))
        elif strict_positive:
            return max(1, value)
        return value
    
    def get_delta_time(phase: str, direction: str = 'both') -> float:
        """
        Get realistic delta time based on phase and direction.
        Adds randomness for network conditions.
        """
        if phase == 'handshake':
            if direction == 'c2s':
                base = get_stat('delta_times', 'handshake_c2s', 0.015, 0.010)
            else:
                base = get_stat('delta_times', 'handshake_s2c', 0.020, 0.015)
        elif phase == 'data':
            if direction == 'c2s':
                base = get_stat('delta_times', 'client_request', 0.010, 0.005)
            else:
                base = get_stat('delta_times', 'server_response', 0.005, 0.003)
        elif phase == 'ack':
            base = get_stat('delta_times', 'ack_response', 0.002, 0.001)
        else:
            base = 0.001
        
        # Add random jitter (10-50% variance)
        jitter = np.random.uniform(0.7, 1.3)
        return max(0.000001, base * jitter)
    
    def get_packet_size(packet_type: str, fallback_mean: int = 100, 
                       fallback_std: int = 20) -> int:
        """
        Get packet size from statistics with protocol-aware fallbacks.
        """
        # Try to get from stats first
        size_key = f'{packet_type}'
        size = get_stat('packet_sizes', size_key, fallback_mean, fallback_std)
        
        # Ensure minimum sizes for specific packet types
        if 'initial' in packet_type.lower():
            return max(1200, int(size))  # Initial packets need to be large
        elif 'path_' in packet_type.lower():
            return int(np.random.uniform(PATH_VALIDATION_MTU_MIN, PATH_VALIDATION_MTU_MAX))
        elif 'ack' in packet_type.lower():
            return max(60, min(150, int(size)))  # ACKs are small
        
        return int(size)
    
    def create_packet(delta: float, length: int, direction: int, header_form: int,
                     **counts) -> dict:
        """
        Create a packet feature dictionary with all fields.
        """
        nonlocal current_packet_count, current_time_msec
        current_packet_count += 1
        current_time_msec += delta
        
        # Default all counts to 0
        packet = {
            'frame_number': current_packet_count,
            'delta_time': round(delta, 6),
            'packet_length': int(length),
            'packet_direction': direction,
            'header_form': header_form,
            'count_initial': 0,
            'count_0rtt': 0,
            'count_handshake': 0,
            'count_1rtt': 0,
            'count_retry': 0,
            'count_vn': 0,
            'count_ack': 0,
            'count_padding': 0,
            'count_connection_close': 0,
            'count_path_challenge': 0,
            'count_path_response': 0,
            'count_new_connection_id': 0,
            'count_retire_cid': 0,
            'count_crypto': 0,
            'count_handshake_done': 0,
            'http3_stream_count': 0,
            'http3_fin_count': 0,
            'stream_length': 0,
            'stream_type_count': 0
        }
        
        # Update with provided counts
        packet.update(counts)
        return packet
    
    # =================================================================
    # PHASE 1: HANDSHAKE
    # =================================================================
    if blueprint.get('retry_occurred', 0) == 1:
        # Retry handshake sequence
        # 1. Client Initial attempt
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 0, count_1rtt=1
        ))
        
        # 2. Server early response (VN or similar)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('ack_server', 95, 15),
            1, 0, count_1rtt=1, count_vn=1
        ))
        
        # 3. Client Initial (after seeing VN)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 4. Server Retry
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_other_server', 137, 20),
            1, 1, count_retry=1
        ))
        
        # 5. Client Initial with retry token
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 6. Server Initial + Handshake (large packet)
        num_crypto = np.random.randint(1, 3)  # Variable crypto frames
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1, 
            count_initial=1, 
            count_handshake=np.random.randint(0, 2),
            count_ack=1, 
            count_crypto=num_crypto
        ))
        
        # 7. Server Handshake continuation (if needed)
        if np.random.random() > 0.3:  # 70% chance of continuation packet
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 8. Client Handshake completion
        num_packet_types = np.random.randint(2, 4)  # Variable packet type mixing
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=1
        ))
    else:
        # Standard handshake (no retry)
        # 1. Client Initial
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 2. Server Initial + Handshake
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1,
            count_initial=1,
            count_handshake=np.random.randint(0, 2),
            count_ack=1,
            count_crypto=np.random.randint(1, 3)
        ))
        
        # 3. Possible Server Handshake continuation
        if np.random.random() > 0.4:
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 4. Client Handshake completion
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=np.random.randint(0, 2)
        ))
    
    # =================================================================
    # PHASE 2: HTTP/3 INITIALIZATION
    # =================================================================
    # Server sends HANDSHAKE_DONE + initial HTTP/3 setup
    settings_size = np.random.randint(15, 25)  # Variable SETTINGS size
    output_packet_features.append(create_packet(
        get_delta_time('data', 's2c'),
        get_packet_size('pkt_size_uni_server', 556, 100),
        1, 0,
        count_1rtt=1,
        count_ack=np.random.randint(0, 2),
        count_new_connection_id=np.random.randint(0, 2),
        count_crypto=np.random.randint(0, 2),
        count_handshake_done=1,
        http3_stream_count=1,
        stream_length=settings_size,
        stream_type_count=1
    ))
    server_app_bytes_sent += settings_size
    
    # Server sends unidirectional control streams
    server_uni_count = blueprint.get('server_uni_streams_count', 4)
    for i in range(max(0, server_uni_count - 1)):
        stream_len = np.random.choice([1, 1, 1, 26, 72], p=[0.5, 0.2, 0.1, 0.1, 0.1])
        is_fin = (i >= server_uni_count - 2) or (np.random.random() > 0.7)
        
        base_size = get_packet_size('pkt_size_uni_server', 92, 20)
        size = base_size + stream_len if stream_len > 1 else base_size
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 's2c'),
            size, 1, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        server_app_bytes_sent += stream_len
    
    # Client ACK
    output_packet_features.append(create_packet(
        get_delta_time('ack'),
        get_packet_size('ack_client', 91, 10),
        0, 0, count_1rtt=1, count_ack=1
    ))
    
    # =================================================================
    # PHASE 3: PRE-MIGRATION PROBING (if applicable)
    # =================================================================
    if blueprint.get('migration_type', 'NONE') != 'NONE':
        # Optional pre-migration path validation
        if np.random.random() > 0.5:  # 50% chance of early probing
            output_packet_features.append(create_packet(
                get_delta_time('data', 'c2s'),
                get_packet_size('path_challenge', 1398, 50),
                0, 0, count_1rtt=1, count_padding=1, count_path_challenge=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('path_response', 1441, 50),
                1, 0, count_1rtt=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 100, 15),
                0, 0, count_1rtt=1, count_ack=1, count_path_response=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    # =================================================================
    # PHASE 4: CLIENT APPLICATION DATA
    # =================================================================
    # Client sends unidirectional streams (QPACK, etc.)
    client_uni_count = blueprint.get('client_uni_streams_count', 4)
    for i in range(client_uni_count):
        # Variable stream sizes
        if i == 0:
            stream_len = np.random.randint(15, 25)  # SETTINGS-like
        elif i >= client_uni_count - 2:
            stream_len = np.random.choice([26, 72], p=[0.6, 0.4])  # Larger final streams
        else:
            stream_len = np.random.randint(1, 5)  # Small control streams
        
        is_fin = (i >= client_uni_count - 2) or (np.random.random() > 0.6)
        base_size = get_packet_size('pkt_size_uni_client', 110, 30)
        size = base_size + (stream_len if stream_len > 10 else 0)
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s'),
            size, 0, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        client_app_bytes_sent += stream_len
    
    # Client sends bidirectional requests
    client_bidi_count = blueprint.get('client_bidi_streams_count', 1)
    avg_request_size = blueprint.get('avg_request_size', 100)
    
    for i in range(client_bidi_count):
        # Variable request sizes around the average
        request_bytes = int(np.random.normal(avg_request_size, avg_request_size * 0.3))
        request_bytes = max(50, request_bytes)
        
        # Maybe piggyback ACK
        has_ack = np.random.random() > 0.5
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s' if i == 0 else 's2c'),
            get_packet_size('pkt_size_bidi_client', 211, 50) + (request_bytes // 10),
            1 if i > 0 else 0, 0,  # Alternate direction sometimes
            count_1rtt=1,
            count_ack=1 if has_ack else 0,
            http3_stream_count=1,
            http3_fin_count=1,
            stream_length=request_bytes,
            stream_type_count=1
        ))
        client_app_bytes_sent += request_bytes
    
    # =================================================================
    # PHASE 5: CONNECTION MIGRATION
    # =================================================================
    migration_type = blueprint.get('migration_type', 'NONE')
    if migration_type != 'NONE' and not has_migrated:
        has_migrated = True
        
        # Wait until migration time
        time_until_migration = blueprint.get('time_to_migration_msec', 50.0) - current_time_msec
        if time_until_migration > 5.0:
            wait_delta = max(0.001, (time_until_migration / 1000.0) * np.random.uniform(0.8, 1.2))
        else:
            wait_delta = get_delta_time('data', 'c2s')
        
        # 1. Client PATH_CHALLENGE on new path (RFC 9000: padded to >= 1200 bytes)
        output_packet_features.append(create_packet(
            wait_delta,
            get_packet_size('path_challenge', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0, count_1rtt=1, count_path_challenge=1, count_padding=1
        ))
        
        # 2. Server PATH_RESPONSE + PATH_CHALLENGE (bidirectional validation)
        validation_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 1000.0
        half_rtt = (validation_rtt / 2.0) * np.random.uniform(0.8, 1.2)
        
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 150, 100),
            1, 0,
            count_1rtt=1,
            count_path_response=1,
            count_path_challenge=1,
            count_padding=1,
            count_ack=np.random.randint(0, 2)
        ))
        
        # 3. Client PATH_RESPONSE
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0,
            count_1rtt=1,
            count_ack=1,
            count_path_response=1,
            count_padding=1
        ))
        
        # 4. Server ACK
        output_packet_features.append(create_packet(
            get_delta_time('ack'),
            get_packet_size('ack_server', 91, 10),
            1, 0, count_1rtt=1, count_ack=1
        ))
    
    # =================================================================
    # PHASE 6: POST-MIGRATION DATA
    # =================================================================
    total_server_bytes = blueprint.get('total_server_app_bytes', 0)
    remaining_server_bytes = total_server_bytes - server_app_bytes_sent
    
    if remaining_server_bytes > 20:
        # Send remaining data in variable-sized packets
        avg_response_size = blueprint.get('avg_response_size', 71)
        num_response_packets = max(1, int(remaining_server_bytes / avg_response_size))
        
        for i in range(num_response_packets):
            bytes_this_packet = min(
                int(np.random.normal(avg_response_size, avg_response_size * 0.4)),
                remaining_server_bytes
            )
            bytes_this_packet = max(10, bytes_this_packet)
            
            is_fin = (i == num_response_packets - 1) or (remaining_server_bytes <= bytes_this_packet)
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('pkt_size_bidi_server', 100, 40) + bytes_this_packet,
                1, 0,
                count_1rtt=1,
                http3_stream_count=1 if is_fin else 0,
                http3_fin_count=1 if is_fin else 0,
                stream_length=bytes_this_packet,
                stream_type_count=1 if is_fin else 0
            ))
            
            server_app_bytes_sent += bytes_this_packet
            remaining_server_bytes -= bytes_this_packet
            
            if remaining_server_bytes <= 0:
                break
    
    # =================================================================
    # PHASE 7: CONNECTION CLOSE
    # =================================================================
    close_type = blueprint.get('connection_close_type', 'CLIENT_CLOSE')
    
    # Wait until connection duration
    time_until_close = blueprint.get('connection_duration_msec', 100.0) - current_time_msec
    if time_until_close > 10.0:
        close_delta = max(0.010, (time_until_close / 1000.0) * np.random.uniform(0.9, 1.1))
    else:
        close_delta = get_delta_time('data', 'c2s')
    
    if close_type == 'CLIENT_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_client', 97, 15),
            0, 0, count_1rtt=1, count_connection_close=1
        ))
        
        # Server may ACK (not always captured)
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    elif close_type == 'SERVER_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_server', 97, 15),
            1, 0, count_1rtt=1, count_connection_close=1
        ))
        
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 91, 10),
                0, 0, count_1rtt=1, count_ack=1
            ))
    
    return output_packet_features


# Example usage
if __name__ == "__main__":
    real_world_blueprint = {
        "connection_duration_msec": 107.25, 
        "retry_occurred": 1, 
        "server_issued_cid_count": 1, 
        "migration_type": "IP_AND_PORT", 
        "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 77.66, 
        "total_client_app_bytes": 119, 
        "total_server_app_bytes": 355, 
        "avg_request_size": 23.8, 
        "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, 
        "client_uni_streams_count": 4, 
        "server_uni_streams_count": 4, 
        "time_to_migration_msec": 86.83, 
        "app_data_bytes_before_migration": 47, 
        "migration_validation_duration_msec": 1.17
    }
    
    example_stats = {
        'packet_sizes': {
            'handshake_initial_client': {'mean': 1248, 'std': 10, 'samples': 100},
            'handshake_other_server': {'mean': 517, 'std': 100, 'samples': 50},
            'handshake_other_client': {'mean': 1398, 'std': 50, 'samples': 50},
            'pkt_size_uni_server': {'mean': 100, 'std': 30, 'samples': 200},
            'pkt_size_uni_client': {'mean': 110, 'std': 25, 'samples': 200},
            'pkt_size_bidi_client': {'mean': 211, 'std': 40, 'samples': 100},
            'pkt_size_bidi_server': {'mean': 150, 'std': 50, 'samples': 200},
            'ack_client': {'mean': 91, 'std': 8, 'samples': 500},
            'ack_server': {'mean': 91, 'std': 8, 'samples': 500},
            'path_challenge': {'mean': 1398, 'std': 30, 'samples': 50},
            'path_response': {'mean': 1398, 'std': 30, 'samples': 50},
            'close_client': {'mean': 97, 'std': 10, 'samples': 50},
        },
        'delta_times': {
            'handshake_s2c': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'handshake_c2s': {'mean': 0.0020, 'std': 0.0015, 'samples': 100},
            'server_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 1000},
            'client_request': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'ack_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 500}
        }
    }
    
    # Generate 3 different captures from the same blueprint
    print("Generating 3 varied captures from the same blueprint...\n")
    
    for run in range(3):
        packets = generate_statistically_realistic_features(real_world_blueprint, example_stats)
        
        print(f"=== RUN {run + 1} ===")
        print(f"Generated {len(packets)} packets")
        print(f"Sample packets:")
        for pkt in packets[:3]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print(f"  ... (middle packets)")
        for pkt in packets[-2:]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print()
        
        # Save to CSV
        df = pd.DataFrame(packets)
        filename = f'generated_capture_run{run + 1}.csv'
        df.to_csv(filename, index=False)
        print(f"Saved to {filename}\n")

Generating 3 varied captures from the same blueprint...

=== RUN 1 ===
Generated 28 packets
Sample packets:
  #1: Δ0.000000s, 1247B, dir=0
  #2: Δ0.001086s, 94B, dir=1
  #3: Δ0.001564s, 1248B, dir=0
  ... (middle packets)
  #27: Δ0.097225s, 109B, dir=0
  #28: Δ0.000321s, 102B, dir=1

Saved to generated_capture_run1.csv

=== RUN 2 ===
Generated 28 packets
Sample packets:
  #1: Δ0.000000s, 1237B, dir=0
  #2: Δ0.000240s, 87B, dir=1
  #3: Δ0.001548s, 1230B, dir=0
  ... (middle packets)
  #27: Δ0.115783s, 96B, dir=0
  #28: Δ0.000004s, 81B, dir=1

Saved to generated_capture_run2.csv

=== RUN 3 ===
Generated 26 packets
Sample packets:
  #1: Δ0.000000s, 1254B, dir=0
  #2: Δ0.000139s, 104B, dir=1
  #3: Δ0.000673s, 1237B, dir=0
  ... (middle packets)
  #25: Δ0.000556s, 186B, dir=1
  #26: Δ0.103536s, 99B, dir=0

Saved to generated_capture_run3.csv



In [ ]:
real_world_blueprint = {
        "connection_duration_msec": 42.25, "retry_occurred": 1, "server_issued_cid_count": 1, "migration_type": "IP_AND_PORT", "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 12.66, "total_client_app_bytes": 119, "total_server_app_bytes": 355, "avg_request_size": 23.8, "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, "client_uni_streams_count": 4, "server_uni_streams_count": 4, "time_to_migration_msec": 86.83, "app_data_bytes_before_migration": 47, "migration_validation_duration_msec": 1.17
    }

print("--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---")
low_level_packets = generate_statistically_realistic_features(real_world_blueprint, full_stats_profile)
df = pd.DataFrame(low_level_packets)

display(df)

--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---


,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,...,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000000,1240,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0.003557,66,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0.001530,1265,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,4,0.001694,60,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,0.000001,1211,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
5,6,0.000212,1324,1,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
6,7,0.005646,63,1,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
7,8,0.002740,1065,0,1,1,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
8,9,0.004113,279,1,0,0,0,0,1,0,...,0,0,0,0,0,1,1,0,24,1
9,10,0.012239,357,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,1


## Version 2

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Dict, Any

In [ ]:
class TrafficProfiler:
    def __init__(self, high_level_df: pd.DataFram, low_level_df: pd.DataFrame):
        self.hl_df = high_level_df.copy()
        self.ll_df = low_level_df.copy()
        self.stat_profiles = {}

    def build_profiles(self) -> Dict[str, Any]:
        """Build statistical profiles from high-level and low-level dataframes."""
        # to know which low level dataset is which implementation
        merged = pd.merge(
            self.ll_df, 
            self.hl_df[['file_id', 'implementation']], 
            on='file_id', 
            how='inner'
        )
        # get the unique implementations
        implementations = merged['implementation'].unique()

        for impl in implementations:
            print(f"\nBuilding detailed profile for implementation: {impl}")
            df = merged[merged['implementation'] == impl].copy()

            raw_stats = {
                'packet_sizes': defaultdict(list),
                'delta_times': defaultdict(list),
                'ack_frequency': defaultdict(list),
                'behavior_counts': {}
            }

            # Direction: 0 = Client->Server, 1 = Server->Client
            is_client = df['packet_direction'] == 0
            is_server = df['packet_direction'] == 1
            is_long_header = df['header_form'] == 1 

            # ------ Handshake statistics ------
            hs_df = df[is_long_header] # packets with long header are found at the start of the connection, Initial and Handshake packets

            #Initial packets
            mask_initial_client = (hs_df['packet_direction'] == 0) & (hs_df['count_initial'] > 0)
            raw_stats['packet_sizes']['handshake_initial_client'] += hs_df.loc[
                mask_initial_client, 'packet_length'
            ].tolist()

            mask_initial_server = (hs_df['packet_direction'] == 1) & (hs_df['count_initial'] > 0)
            raw_stats['packet_sizes']['handshake_initial_server'] = hs_df.loc[
                mask_initial_server, 'packet_length'
            ].tolist()

            #Other handshake packets
            raw_stats['packet_sizes']['handshake_other_server'] = hs_df.loc[
                (hs_df['packet_direction'] == 1) & (hs_df['count_handshake'] > 0), 'packet_length'
            ].tolist()
            
            raw_stats['packet_sizes']['handshake_other_client'] = hs_df.loc[
                (hs_df['packet_direction'] == 0) & (hs_df['count_handshake'] > 0), 'packet_length'
            ].tolist()

            raw_stats['delta_times']['handshake_c2s'] = hs_df.loc[hs_df['packet_direction'] == 0, 'delta_time'].tolist()
            raw_stats['delta_times']['handshake_s2c'] = hs_df.loc[hs_df['packet_direction'] == 1, 'delta_time'].tolist()

            # ------ 1RTT statistics ------
            rtt_df = df[~is_long_header].copy()

            # Migration behavior
            mask_pc = rtt_df['count_path_challenge'] > 0
            raw_stats['packet_sizes']['path_challenge'] = rtt_df.loc[mask_pc, 'packet_length'].tolist()
            
            mask_pr = rtt_df['count_path_response'] > 0
            raw_stats['packet_sizes']['path_response'] = rtt_df.loc[mask_pr, 'packet_length'].tolist()
            
            # some implementations add padding to path validation and path response packets, some not
            avg_pc_size = rtt_df.loc[mask_pc, 'packet_length'].mean() if len(rtt_df.loc[mask_pc]) > 0 else 0
            raw_stats['behavior']['padded_validation'] = avg_pc_size > 1000

            # packets with only ACK frames - no stream data or migration or connection close frames
            mask_ack_only = (
                (rtt_df['count_ack'] > 0) &
                (rtt_df['count_path_challenge'] == 0) &
                (rtt_df['count_path_response'] == 0) &
                (rtt_df['count_connection_close'] == 0) &
                (rtt_df['http3_stream_count'] == 0)
            )

            raw_stats['packet_sizes']['ack_client'] = rtt_df.loc[mask_ack_only & is_client, 'packet_length'].tolist()
            raw_stats['packet_sizes']['ack_server'] = rtt_df.loc[mask_ack_only & is_server, 'packet_length'].tolist()
            raw_stats['delta_times']['ack_response'] = rtt_df.loc[mask_ack_only, 'delta_time'].tolist()

            # Stream data statistics
            mask_stream = rtt_df['stream_length'] > 0

            # when a client sends stream data that is a request

            raw_stats['packet_sizes']['pkt_size_bidi_client'] = rtt_df.loc[mask_stream & is_client, 'packet_length'].tolist()
            raw_stats['delta_times']['client_request'] = rtt_df.loc[mask_stream & is_client, 'delta_time'].tolist()

            

